In [49]:
from pathlib import Path
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix
)
from sklearn.metrics import precision_recall_curve
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, adjusted_rand_score
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.inspection import permutation_importance


### 1. DATA Loading

In [50]:
data_path = Path("/Users/chunghyunhan/Projects/JPM_TakeHomeProject/census-bureau.data")
cols_path = Path("/Users/chunghyunhan/Projects/JPM_TakeHomeProject/census-bureau.columns")

cols = [c.strip() for c in cols_path.read_text().splitlines() if c.strip()]
print("num_cols:", len(cols))

df = pd.read_csv(data_path, header=None, names=cols,low_memory=False)

num_cols: 42


In [51]:
df.head(5)

,age,class of worker,detailed industry recode,detailed occupation recode,education,wage per hour,enroll in edu inst last wk,marital stat,major industry code,major occupation code,race,hispanic origin,sex,member of a labor union,reason for unemployment,full or part time employment stat,capital gains,capital losses,dividends from stocks,tax filer stat,region of previous residence,state of previous residence,detailed household and family stat,detailed household summary in household,weight,migration code-change in msa,migration code-change in reg,migration code-move within reg,live in this house 1 year ago,migration prev res in sunbelt,num persons worked for employer,family members under 18,country of birth father,country of birth mother,country of birth self,citizenship,own business or self employed,fill inc questionnaire for veteran's admin,veterans benefits,weeks worked in year,year,label
0,73,Not in universe,0,0,High school graduate,0,Not in universe,Widowed,Not in universe or children,Not in universe,White,All other,Female,Not in universe,Not in universe,Not in labor force,0,0,0,Nonfiler,Not in universe,Not in universe,Other Rel 18+ ever marr not in subfamily,Other relative of householder,1700.09,?,?,?,Not in universe under 1 year old,?,0,Not in universe,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,2,0,95,- 50000.
1,58,Self-employed-not incorporated,4,34,Some college but no degree,0,Not in universe,Divorced,Construction,Precision production craft & repair,White,All other,Male,Not in universe,Not in universe,Children or Armed Forces,0,0,0,Head of household,South,Arkansas,Householder,Householder,1053.55,MSA to MSA,Same county,Same county,No,Yes,1,Not in universe,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,2,52,94,- 50000.
2,18,Not in universe,0,0,10th grade,0,High school,Never married,Not in universe or children,Not in universe,Asian or Pacific Islander,All other,Female,Not in universe,Not in universe,Not in labor force,0,0,0,Nonfiler,Not in universe,Not in universe,Child 18+ never marr Not in a subfamily,Child 18 or older,991.95,?,?,?,Not in universe under 1 year old,?,0,Not in universe,Vietnam,Vietnam,Vietnam,Foreign born- Not a citizen of U S,0,Not in universe,2,0,95,- 50000.
3,9,Not in universe,0,0,Children,0,Not in universe,Never married,Not in universe or children,Not in universe,White,All other,Female,Not in universe,Not in universe,Children or Armed Forces,0,0,0,Nonfiler,Not in universe,Not in universe,Child <18 never marr not in subfamily,Child under 18 never married,1758.14,Nonmover,Nonmover,Nonmover,Yes,Not in universe,0,Both parents present,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,0,0,94,- 50000.
4,10,Not in universe,0,0,Children,0,Not in universe,Never married,Not in universe or children,Not in universe,White,All other,Female,Not in universe,Not in universe,Children or Armed Forces,0,0,0,Nonfiler,Not in universe,Not in universe,Child <18 never marr not in subfamily,Child under 18 never married,1069.16,Nonmover,Nonmover,Nonmover,Yes,Not in universe,0,Both parents present,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,0,0,94,- 50000.


In [71]:
# Sanity Check
print("DataFrame shape:", df.shape)
# 1) Check field count 
expected = len(cols)
bad_lines = []
with data_path.open("r") as f:
    for idx, line in enumerate(f, start=1):
        if len(line.rstrip("\n").split(",")) != expected:
            bad_lines.append(idx)

print("All lines have correct field count:", len(bad_lines) == 0)
if bad_lines:
    print("Bad line indices (first 10):", bad_lines[:10])

# 2) Dtype distribution
print("\nDtype counts:")
print(df.dtypes.value_counts())

# 3) Actual NaN counts (top 10)
print("\nTop NA columns:")
print(df.isna().sum().sort_values(ascending=False).head(5))

# 4) Sentinel values (sample of categorical columns)
sample_cols = df.select_dtypes(include=["object", "string"]).columns[:]
print("\nSample sentinel values:")
for col in sample_cols:
    print(col, df[col].value_counts().head(3).to_dict())

print("\nLabel distribution:")
print(df["label"].value_counts())
print("\nLabel ratio:")
print(df["label"].value_counts(normalize=True))

DataFrame shape: (199523, 48)
All lines have correct field count: True

Dtype counts:
str        30
int64      12
float64     5
int32       1
Name: count, dtype: int64

Top NA columns:
hispanic origin                   874
age                                 0
own business or self employed       0
migration code-move within reg      0
live in this house 1 year ago       0
dtype: int64

Sample sentinel values:
class of worker {'Not in universe': 100245, 'Private': 72028, 'Self-employed-not incorporated': 8445}
education {'High school graduate': 48407, 'Children': 47422, 'Some college but no degree': 27820}
enroll in edu inst last wk {'Not in universe': 186943, 'High school': 6892, 'College or university': 5688}
marital stat {'Never married': 86485, 'Married-civilian spouse present': 84222, 'Divorced': 12710}
major industry code {'Not in universe or children': 100684, 'Retail trade': 17070, 'Manufacturing-durable goods': 9015}
major occupation code {'Not in universe': 100684, 'Adm suppor

### 2. Preprocessing and Train/Validation Split

In this step, I prepared the dataset for modeling and created a reliable evaluation split.

1. **Target and sample weights**
   - The target `label` was converted into a binary variable:  
     `50000+.` → 1, `- 50000.` → 0.  
   - The `weight` column was separated for later use as `sample_weight` during training to reflect the survey’s population weighting.

2. **Feature separation**
   - The remaining 40 variables were treated as input features `X`.
   - Numeric variables were split into two groups:
     - **Continuous features** (scaled): `age`, `wage per hour`, `capital gains`, `capital losses`, `dividends from stocks`, `weeks worked in year`.
     - **Discrete numeric codes** (treated as categorical):  
       `detailed industry recode`, `detailed occupation recode`, `num persons worked for employer`,  
       `own business or self employed`, `veterans benefits`, `year`.

3. **Missing values**
   - For numeric features, missing values are imputed with the median.
   - For categorical features, missing values are imputed with the most frequent category.
   - Values such as “Not in universe” are treated as valid categories rather than missing data.

4. **Encoding and scaling**
   - Continuous numeric features are standardized with `StandardScaler`.
   - Categorical features (both string and discrete numeric codes) are one‑hot encoded.
   - Very rare categories are grouped using `min_frequency` to reduce sparsity and improve model convergence.

5. **Train/validation split**
   - The dataset is split into 80% training and 20% validation.
   - The split is **stratified** to preserve the class imbalance (~6% positive class) in both sets.

This preprocessing pipeline ensures that the model receives clean, consistent inputs and that evaluation results are representative of real performance.


In [53]:
# target / weight
target_col = "label"
weight_col = "weight"

y = (df[target_col].str.strip() == "50000+.").astype(int)
sample_weight = df[weight_col].values

X = df.drop(columns=[target_col, weight_col])

# numeric columns (continuous)
continuous_cols = [
    "age",
    "wage per hour",
    "capital gains",
    "capital losses",
    "dividends from stocks",
    "weeks worked in year",
]

# numeric but categorical in nature
categorical_numeric_cols = [
    "detailed industry recode",
    "detailed occupation recode",
    "num persons worked for employer",
    "own business or self employed",
    "veterans benefits",
    "year",
]

# string categorical
categorical_str_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()

categorical_cols = categorical_str_cols + categorical_numeric_cols

# ensure categorical numeric are treated as strings
for c in categorical_numeric_cols:
    X[c] = X[c].astype("string")

# preprocess
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, continuous_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)

# train/valid split (stratify for imbalance)
X_train, X_valid, y_train, y_valid, w_train, w_valid = train_test_split(
    X, y, sample_weight, test_size=0.2, random_state=42, stratify=y
)

print("X_train.shape, X_valid.shape:", X_train.shape, X_valid.shape)
print(y_train.mean(), y_valid.mean())  # positive rate check


X_train.shape, X_valid.shape: (159618, 40) (39905, 40)
0.06206066984926512 0.06204736248590402


### 3. Baseline Classification Model and Evaluation

In this step, I trained a simple baseline classifier to establish a performance benchmark before trying more complex models. I used **logistic regression** because it is fast, interpretable, and handles high‑dimensional one‑hot encoded features well.

1. **Model choice**
   - Logistic Regression with the `saga` solver and `max_iter=200`.
   - This provides a strong linear baseline and scales efficiently with sparse features.

2. **Pipeline integration**
   - The model is wrapped in a single pipeline with the preprocessing transformer from Step 2.
   - This ensures all transformations are applied consistently at training and evaluation time.

3. **Sample weighting**
   - The survey `weight` is passed as `sample_weight` during model fitting and metric computation.
   - This aligns evaluation with the true population distribution.

4. **Evaluation metrics**
   - **ROC‑AUC** and **PR‑AUC** to capture ranking quality under class imbalance.
   - **Precision, Recall, F1, Accuracy**, and **Confusion Matrix** for threshold‑based performance.

This baseline provides a reference point for later comparisons with more complex models.


In [54]:
# 1) Tune SGD Logistic Regression (fast baseline)
alphas = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
results = []

for a in alphas:
    clf = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=a,
        max_iter=2000,
        tol=1e-3,
        early_stopping=True,
        n_iter_no_change=5,
        random_state=42
    )

    model = Pipeline(steps=[
        ("preprocess", preprocess),
        ("clf", clf)
    ])
    model.fit(X_train, y_train, clf__sample_weight=w_train)

    y_proba = model.predict_proba(X_valid)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    res = {
        "alpha": a,
        "model": model,
        "y_proba": y_proba,
        "roc_auc": roc_auc_score(y_valid, y_proba, sample_weight=w_valid),
        "pr_auc": average_precision_score(y_valid, y_proba, sample_weight=w_valid),
        "precision": precision_score(y_valid, y_pred, sample_weight=w_valid),
        "recall": recall_score(y_valid, y_pred, sample_weight=w_valid),
        "f1": f1_score(y_valid, y_pred, sample_weight=w_valid),
        "accuracy": accuracy_score(y_valid, y_pred, sample_weight=w_valid),
    }
    results.append(res)

# Select best model by PR-AUC
best = max(results, key=lambda x: x["pr_auc"])
print("Best alpha:", best["alpha"])
print("ROC-AUC:", best["roc_auc"])
print("PR-AUC :", best["pr_auc"])
print("Precision:", best["precision"])
print("Recall   :", best["recall"])
print("F1       :", best["f1"])
print("Accuracy :", best["accuracy"])

# 2) Threshold tuning: maximize F1
y_proba = best["y_proba"]
precision, recall, thresholds = precision_recall_curve(
    y_valid, y_proba, sample_weight=w_valid
)

f1_scores = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = int(np.nanargmax(f1_scores))

best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
print("Best threshold (F1):", best_threshold)

y_pred_opt = (y_proba >= best_threshold).astype(int)
print("Optimized Precision:", precision_score(y_valid, y_pred_opt, sample_weight=w_valid))
print("Optimized Recall   :", recall_score(y_valid, y_pred_opt, sample_weight=w_valid))
print("Optimized F1       :", f1_score(y_valid, y_pred_opt, sample_weight=w_valid))
print("Optimized Accuracy :", accuracy_score(y_valid, y_pred_opt, sample_weight=w_valid))


Best alpha: 0.01
ROC-AUC: 0.921931266360842
PR-AUC : 0.5259362576450782
Precision: 0.38738539258081506
Recall   : 0.6971974938149385
F1       : 0.4980423870514757
Accuracy : 0.9092569218308924
Best threshold (F1): 0.9414638806112821
Optimized Precision: 0.49777937321850707
Optimized Recall   : 0.570846739658886
Optimized F1       : 0.5318150639349586
Optimized Accuracy : 0.9351017065554234


### 4. Model Tuning and Threshold Optimization

In this step, I improved the baseline model by tuning hyperparameters and selecting a decision threshold that better matches the business goal under class imbalance.

1. **Regularization tuning (Logistic Regression)**
   - I tested multiple values of the regularization strength `C`.
   - Each model was evaluated using **ROC‑AUC** and **PR‑AUC**, with weighted metrics to reflect population weights.
   - The model with the best **PR‑AUC** was selected as the candidate.

2. **Threshold optimization**
   - The default threshold of 0.5 often yields low recall in imbalanced datasets.
   - I computed the precision–recall curve and selected the threshold that maximizes **F1**.
   - This provides a better balance between precision and recall for marketing use cases.

3. **Optional business‑driven threshold**
   - If the client values recall (finding more high‑income customers), the threshold can be lowered.
   - If the client values precision (reducing outreach costs), the threshold can be raised.

This step delivers a stronger, more business‑aligned classifier without adding unnecessary model complexity.


In [55]:
# 1) Tune SGD Logistic Regression (fast baseline)
alphas = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
results = []

for a in alphas:
    clf = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=a,
        max_iter=2000,
        tol=1e-3,
        early_stopping=True,
        n_iter_no_change=5,
        random_state=42
    )

    model = Pipeline(steps=[
        ("preprocess", preprocess),
        ("clf", clf)
    ])
    model.fit(X_train, y_train, clf__sample_weight=w_train)

    y_proba = model.predict_proba(X_valid)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    res = {
        "alpha": a,
        "model": model,
        "y_proba": y_proba,
        "roc_auc": roc_auc_score(y_valid, y_proba, sample_weight=w_valid),
        "pr_auc": average_precision_score(y_valid, y_proba, sample_weight=w_valid),
        "precision": precision_score(y_valid, y_pred, sample_weight=w_valid),
        "recall": recall_score(y_valid, y_pred, sample_weight=w_valid),
        "f1": f1_score(y_valid, y_pred, sample_weight=w_valid),
        "accuracy": accuracy_score(y_valid, y_pred, sample_weight=w_valid),
    }
    results.append(res)

# Select best model by PR-AUC
best = max(results, key=lambda x: x["pr_auc"])
print("Best alpha:", best["alpha"])
print("ROC-AUC:", best["roc_auc"])
print("PR-AUC :", best["pr_auc"])
print("Precision:", best["precision"])
print("Recall   :", best["recall"])
print("F1       :", best["f1"])
print("Accuracy :", best["accuracy"])

# 2) Threshold tuning: maximize F1
y_proba = best["y_proba"]
precision, recall, thresholds = precision_recall_curve(
    y_valid, y_proba, sample_weight=w_valid
)

f1_scores = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = int(np.nanargmax(f1_scores))

best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
print("Best threshold (F1):", best_threshold)

y_pred_opt = (y_proba >= best_threshold).astype(int)
print("Optimized Precision:", precision_score(y_valid, y_pred_opt, sample_weight=w_valid))
print("Optimized Recall   :", recall_score(y_valid, y_pred_opt, sample_weight=w_valid))
print("Optimized F1       :", f1_score(y_valid, y_pred_opt, sample_weight=w_valid))
print("Optimized Accuracy :", accuracy_score(y_valid, y_pred_opt, sample_weight=w_valid))


Best alpha: 0.01
ROC-AUC: 0.921931266360842
PR-AUC : 0.5259362576450782
Precision: 0.38738539258081506
Recall   : 0.6971974938149385
F1       : 0.4980423870514757
Accuracy : 0.9092569218308924
Best threshold (F1): 0.9414638806112821
Optimized Precision: 0.49777937321850707
Optimized Recall   : 0.570846739658886
Optimized F1       : 0.5318150639349586
Optimized Accuracy : 0.9351017065554234


### 5. Tree‑Based Model Comparison (Recall / PR‑AUC Improvement)

To improve recall and PR‑AUC, I compared tree‑based models that can capture non‑linear relationships beyond logistic regression.  
Because one‑hot encoding is inefficient for trees, I used **ordinal encoding** for categorical variables and applied median/mode imputation.

1. **Preprocessing for trees**
   - Numeric features: median imputation (no scaling required for trees).
   - Categorical features: most‑frequent imputation + ordinal encoding.
   - This keeps the feature space compact and suitable for tree models.

2. **Models evaluated**
   - **RandomForestClassifier**  
     Robust baseline for non‑linear patterns and interactions.
   - **HistGradientBoostingClassifier**  
     Gradient‑boosted trees with native handling of categorical features (via encoded indices).

3. **Evaluation**
   - Used the same train/validation split and sample weights.
   - Reported **ROC‑AUC, PR‑AUC, Precision, Recall, F1, Accuracy** to compare against the tuned logistic regression.

This step identifies whether tree‑based models provide meaningful gains in recall and PR‑AUC for the marketing objective.


In [56]:
categorical_cols = categorical_str_cols + categorical_numeric_cols

# ensure categorical numeric are strings (if not already)
for c in categorical_numeric_cols:
    X_train[c] = X_train[c].astype("string")
    X_valid[c] = X_valid[c].astype("string")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

preprocess_tree = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, continuous_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)

# categorical feature indices for HGB
n_num = len(continuous_cols)
cat_feature_indices = list(range(n_num, n_num + len(categorical_cols)))

# ====== models ======
models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=50,
        n_jobs=-1,
        random_state=42
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.1,
        max_depth=None,
        random_state=42,
        categorical_features=cat_feature_indices
    )
}

# ====== train + evaluate ======
results = []
for name, clf in models.items():
    model = Pipeline(steps=[
        ("preprocess", preprocess_tree),
        ("clf", clf)
    ])

    model.fit(X_train, y_train, clf__sample_weight=w_train)

    y_proba = model.predict_proba(X_valid)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    res = {
        "model": name,
        "roc_auc": roc_auc_score(y_valid, y_proba, sample_weight=w_valid),
        "pr_auc": average_precision_score(y_valid, y_proba, sample_weight=w_valid),
        "precision": precision_score(y_valid, y_pred, sample_weight=w_valid),
        "recall": recall_score(y_valid, y_pred, sample_weight=w_valid),
        "f1": f1_score(y_valid, y_pred, sample_weight=w_valid),
        "accuracy": accuracy_score(y_valid, y_pred, sample_weight=w_valid)
    }
    results.append(res)

# print results
for r in results:
    print(r["model"])
    print(" ROC-AUC:", r["roc_auc"])
    print(" PR-AUC :", r["pr_auc"])
    print(" Precision:", r["precision"])
    print(" Recall   :", r["recall"])
    print(" F1       :", r["f1"])
    print(" Accuracy :", r["accuracy"])
    print("-" * 40)

RandomForest
 ROC-AUC: 0.9480641753201777
 PR-AUC : 0.640477675552523
 Precision: 0.8372083225481235
 Recall   : 0.2741334625079749
 F1       : 0.4130265223295379
 Accuracy : 0.9496893950234265
----------------------------------------
HistGradientBoosting
 ROC-AUC: 0.953645052969345
 PR-AUC : 0.6888236186138641
 Precision: 0.7619514032053538
 Recall   : 0.47508328886341594
 F1       : 0.5852550149317401
 Accuracy : 0.9565226861315634
----------------------------------------


### 6. Final Model: HGB + Threshold Tuning + Final Evaluation

This step finalizes the classification solution by selecting **HistGradientBoosting (HGB)** as the best‑performing model and tuning the decision threshold to optimize business‑relevant performance.

1. **Final model selection**
   - HGB was chosen because it delivered the strongest overall results on the validation set, especially **PR‑AUC** and **Recall**.

2. **Threshold tuning**
   - Rather than using the default 0.5 threshold, I computed the precision‑recall curve and selected the threshold that maximizes **F1**, balancing precision and recall.

3. **Final performance reporting**
   - The model outputs **ROC‑AUC, PR‑AUC, Precision, Recall, F1, Accuracy**, and a **Confusion Matrix** using sample weights to reflect population distribution.

In [ ]:
n_num = len(continuous_cols)
cat_feature_indices = list(range(n_num, n_num + len(categorical_cols)))

# ====== Final HGB model ======
hgb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.1,
    max_depth=None,
    random_state=42,
    categorical_features=cat_feature_indices
)

final_model = Pipeline(steps=[
    ("preprocess", preprocess_tree),
    ("clf", hgb)
])

# Fit
final_model.fit(X_train, y_train, clf__sample_weight=w_train)

# Predict probs
y_proba = final_model.predict_proba(X_valid)[:, 1]

# Threshold tuning (maximize F1)
precision, recall, thresholds = precision_recall_curve(y_valid, y_proba, sample_weight=w_valid)

f1_scores = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = int(np.nanargmax(f1_scores))
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

# Final predictions
y_pred = (y_proba >= best_threshold).astype(int)

# Final metrics
print("Best threshold (F1):", best_threshold)
print("ROC-AUC:", roc_auc_score(y_valid, y_proba, sample_weight=w_valid))
print("PR-AUC :", average_precision_score(y_valid, y_proba, sample_weight=w_valid))
print("Precision:", precision_score(y_valid, y_pred, sample_weight=w_valid))
print("Recall   :", recall_score(y_valid, y_pred, sample_weight=w_valid))
print("F1       :", f1_score(y_valid, y_pred, sample_weight=w_valid))
print("Accuracy :", accuracy_score(y_valid, y_pred, sample_weight=w_valid))
print("Confusion Matrix:\n", confusion_matrix(y_valid, y_pred, sample_weight=w_valid))

Best threshold (F1): 0.30908637932187605
ROC-AUC: 0.953645052969345
PR-AUC : 0.6888236186138641
Precision: 0.6229030277196305
Recall   : 0.6268426956221702
F1       : 0.6248666520144509
Accuracy : 0.9514025092949333
Confusion Matrix:
 [[63251701.54000001  1701400.14      ]
 [ 1673043.52        2810437.04000001]]


### 6A. Probability Calibration (Business Readiness)
Calibrate probabilities so the score can be interpreted as income potential and used for ROI decisions.


In [58]:
# Probability calibration
calibrated_model = CalibratedClassifierCV(
    estimator=final_model,
    method="sigmoid",
    cv=3
)

calibrated_model.fit(X_train, y_train, sample_weight=w_train)

y_proba_cal = calibrated_model.predict_proba(X_valid)[:, 1]

print("Brier score (raw):", brier_score_loss(y_valid, y_proba, sample_weight=w_valid))
print("Brier score (cal):", brier_score_loss(y_valid, y_proba_cal, sample_weight=w_valid))
print("ROC-AUC (cal):", roc_auc_score(y_valid, y_proba_cal, sample_weight=w_valid))
print("PR-AUC  (cal):", average_precision_score(y_valid, y_proba_cal, sample_weight=w_valid))


/Users/chunghyunhan/miniconda3/lib/python3.13/site-packages/sklearn/calibration.py:394: UserWarning: Since Pipeline does not appear to accept sample_weight, sample weights will only be used for the calibration itself. This can be caused by a limitation of the current scikit-learn API. See the following issue for more details: https://github.com/scikit-learn/scikit-learn/issues/21134. Be warned that the result of the calibration is likely to be incorrect.
  warnings.warn(


Brier score (raw): 0.03305583948181263
Brier score (cal): 0.03264230074076737
ROC-AUC (cal): 0.9551893025469567
PR-AUC  (cal): 0.6982890921643787


### 6B. ROI-Based Threshold Selection
Choose a threshold that maximizes expected value under marketing cost and conversion assumptions.


In [59]:
# ROI-based threshold selection (edit these assumptions)
cost_per_contact = 5.0
margin_per_customer = 500.0
conversion_rate = 0.05

value_per_tp = margin_per_customer * conversion_rate

proba_for_roi = y_proba_cal if "y_proba_cal" in locals() else y_proba

thresholds = np.linspace(0.0, 1.0, 201)
best_value = -np.inf
best_thr = 0.5

for t in thresholds:
    pred = proba_for_roi >= t
    tp = w_valid[(y_valid == 1) & pred].sum()
    fp = w_valid[(y_valid == 0) & pred].sum()

    expected_value = tp * value_per_tp - (tp + fp) * cost_per_contact

    if expected_value > best_value:
        best_value = expected_value
        best_thr = t

roi_pred = (proba_for_roi >= best_thr).astype(int)

print("ROI-optimal threshold:", best_thr)
print("Expected value (population-weighted):", best_value)
print("Precision:", precision_score(y_valid, roi_pred, sample_weight=w_valid))
print("Recall   :", recall_score(y_valid, roi_pred, sample_weight=w_valid))
print("F1       :", f1_score(y_valid, roi_pred, sample_weight=w_valid))


ROI-optimal threshold: 0.23500000000000001
Expected value (population-weighted): 49980153.65
Precision: 0.5531404450776878
Recall   : 0.6984418306477495
F1       : 0.6173568171400388


### 6C. Weighted Cross-Validation (Robustness Check)
Evaluate stability across folds using weighted ROC‑AUC and PR‑AUC.


In [60]:
# Weighted CV for HGB (3-fold for speed)
X_cv = X.copy()
for c in categorical_numeric_cols:
    X_cv[c] = X_cv[c].astype("string")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_results = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cv, y), start=1):
    X_tr, X_va = X_cv.iloc[tr_idx], X_cv.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    w_tr, w_va = sample_weight[tr_idx], sample_weight[va_idx]

    model = Pipeline(steps=[
        ("preprocess", preprocess_tree),
        ("clf", HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.1,
            max_depth=None,
            random_state=42,
            categorical_features=cat_feature_indices
        ))
    ])

    model.fit(X_tr, y_tr, clf__sample_weight=w_tr)
    y_proba_cv = model.predict_proba(X_va)[:, 1]

    cv_results.append({
        "fold": fold,
        "roc_auc": roc_auc_score(y_va, y_proba_cv, sample_weight=w_va),
        "pr_auc": average_precision_score(y_va, y_proba_cv, sample_weight=w_va)
    })

cv_df = pd.DataFrame(cv_results)
print(cv_df)
print("ROC-AUC mean/std:", cv_df["roc_auc"].mean(), cv_df["roc_auc"].std())
print("PR-AUC  mean/std:", cv_df["pr_auc"].mean(), cv_df["pr_auc"].std())


   fold   roc_auc    pr_auc
0     1  0.951401  0.664420
1     2  0.952639  0.679903
2     3  0.950615  0.675509
ROC-AUC mean/std: 0.9515518049574195 0.0010206816925447944
PR-AUC  mean/std: 0.6732770109284175 0.00797925789097975


### 6D. Feature Importance (Permutation)
Estimate which variables contribute most to predictive performance.


In [61]:
# Permutation importance on a validation sample (unweighted)
feature_names = continuous_cols + categorical_cols

# sample for speed
rng = np.random.RandomState(42)
idx = rng.choice(len(X_valid), size=min(20000, len(X_valid)), replace=False)
X_imp = X_valid.iloc[idx]

y_imp = y_valid.iloc[idx] if hasattr(y_valid, "iloc") else y_valid[idx]

perm = permutation_importance(
    final_model,
    X_imp,
    y_imp,
    n_repeats=5,
    random_state=42,
    scoring="average_precision"
)

imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

print(imp_df.head(15))


                                    feature  importance
0                                       age    0.168883
38                        veterans benefits    0.098775
16                  reason for unemployment    0.096118
4                     dividends from stocks    0.090554
3                            capital losses    0.090321
19             region of previous residence    0.082878
12                                     race    0.067154
22  detailed household summary in household    0.041948
18                           tax filer stat    0.030382
29                  country of birth father    0.028785
17        full or part time employment stat    0.023590
2                             capital gains    0.019394
1                             wage per hour    0.007349
7                                 education    0.005435
5                      weeks worked in year    0.003425


## Task 2

In [62]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from IPython.display import display

### 1A. Cluster Profiling and Interpretation
Summarize cluster size and key economic indicators. Include medians/log transforms for skewed variables to improve interpretability.


In [63]:
# Guard: ensure cluster labels exist
if "cluster" not in df.columns:
    print("cluster not found — creating it now.")
    # prefer best_k if available, else final_k, else default to 3
    if "best_k" in globals():
        final_k = best_k
    elif "final_k" in globals():
        final_k = final_k
    else:
        final_k = 3
        print("best_k not found — defaulting final_k=3")

    kmeans = KMeans(n_clusters=final_k, random_state=42, n_init="auto")
    df["cluster"] = kmeans.fit_predict(X_emb, sample_weight=w_full)

# Guard: ensure income_prob exists
if "income_prob" not in df.columns:
    if "final_model" not in globals():
        raise ValueError("final_model not found. Run the modeling section before segmentation.")
    X_full = df.drop(columns=["label", "weight"])
    for c in categorical_numeric_cols:
        X_full[c] = X_full[c].astype("string")
    df["income_prob"] = final_model.predict_proba(X_full)[:, 1]

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# log transforms for skewed numeric variables
for col in ["wage per hour", "capital gains", "dividends from stocks"]:
    df[f"log_{col}"] = np.log1p(df[col])

# weighted mean helper
def wmean(x, w):
    return np.average(x, weights=w)

# weighted median helper
def weighted_median(values, weights):
    s = pd.DataFrame({"v": values, "w": weights}).sort_values("v")
    cum_w = s["w"].cumsum()
    cutoff = s["w"].sum() / 2.0
    return s.loc[cum_w >= cutoff, "v"].iloc[0]

# summary table
summary = df.groupby("cluster").apply(lambda g: pd.Series({
    "n": g.shape[0],
    "weight_sum": g["weight"].sum(),
    "weight_pct": g["weight"].sum() / df["weight"].sum(),
    "income_prob_mean": wmean(g["income_prob"], g["weight"]),
    "age_mean": wmean(g["age"], g["weight"]),
    "weeks_worked_mean": wmean(g["weeks worked in year"], g["weight"]),
    "wage_per_hour_mean": wmean(g["wage per hour"], g["weight"]),
    "wage_per_hour_median": weighted_median(g["wage per hour"], g["weight"]),
    "capital_gains_mean": wmean(g["capital gains"], g["weight"]),
    "capital_gains_median": weighted_median(g["capital gains"], g["weight"]),
    "dividends_mean": wmean(g["dividends from stocks"], g["weight"]),
    "dividends_median": weighted_median(g["dividends from stocks"], g["weight"]),
    "log_wage_mean": wmean(g["log_wage per hour"], g["weight"]),
    "log_capital_gains_mean": wmean(g["log_capital gains"], g["weight"]),
    "log_dividends_mean": wmean(g["log_dividends from stocks"], g["weight"])
})).sort_values("weight_sum", ascending=False)

summary_fmt = summary.copy().round(3)
summary_fmt["weight_pct"] = (summary_fmt["weight_pct"] * 100).map(lambda x: f"{x:.2f}%")

display(summary_fmt)

# top categories by weight
def top_cats(df, col, top_n=3):
    tmp = (
        df.groupby(["cluster", col])["weight"].sum()
          .groupby(level=0)
          .apply(lambda s: (s / s.sum()).sort_values(ascending=False).head(top_n))
    )
    out = {}
    for cluster in tmp.index.get_level_values(0).unique():
        s = tmp.loc[cluster]
        out[cluster] = ", ".join([f"{idx} ({val*100:.1f}%)" for idx, val in s.items()])
    return out

key_cat_cols = [
    "education", "class of worker", "marital stat",
    "race", "sex", "major industry code", "major occupation code"
]

topcat_table = pd.DataFrame({
    col: pd.Series(top_cats(df, col)) for col in key_cat_cols
})

display(topcat_table)


cluster not found — creating it now.


,n,weight_sum,weight_pct,income_prob_mean,age_mean,weeks_worked_mean,wage_per_hour_mean,wage_per_hour_median,capital_gains_mean,capital_gains_median,dividends_mean,dividends_median,log_wage_mean,log_capital_gains_mean,log_dividends_mean
cluster,,,,,,,,,,,,,,,
1,154030.0,2.707463e+08,78.00%,0.067,34.502,21.662,0.370,0.0,468.807,0.0,222.830,0.0,0.007,0.330,0.666
0,35144.0,5.786829e+07,16.70%,0.051,33.603,25.479,13.051,0.0,355.632,0.0,105.741,0.0,0.153,0.271,0.374
2,10349.0,1.863130e+07,5.40%,0.055,36.768,45.605,1023.261,850.0,312.404,0.0,114.773,0.0,6.798,0.367,0.556


,education,class of worker,marital stat,race,sex,major industry code,major occupation code
0,"(0, 'High school graduate') (23.0%), (0, 'Chil...","(0, 'Private') (45.2%), (0, 'Not in universe')...","(0, 'Never married') (41.5%), (0, 'Married-civ...","(0, 'White') (74.0%), (0, 'Black') (11.0%), (0...","(0, 'Female') (50.7%), (0, 'Male') (49.3%)","(0, 'Not in universe or children') (43.3%), (0...","(0, 'Not in universe') (43.3%), (0, 'Other ser..."
1,"(1, 'Children') (25.9%), (1, 'High school grad...","(1, 'Not in universe') (53.9%), (1, 'Private')...","(1, 'Never married') (45.1%), (1, 'Married-civ...","(1, 'White') (84.8%), (1, 'Black') (13.1%), (1...","(1, 'Female') (51.4%), (1, 'Male') (48.6%)","(1, 'Not in universe or children') (54.2%), (1...","(1, 'Not in universe') (54.2%), (1, 'Professio..."
2,"(2, 'High school graduate') (39.7%), (2, 'Some...","(2, 'Private') (86.1%), (2, 'Local government'...","(2, 'Married-civilian spouse present') (51.9%)...","(2, 'White') (83.0%), (2, 'Black') (13.8%), (2...","(2, 'Male') (50.1%), (2, 'Female') (49.9%)","(2, 'Retail trade') (20.2%), (2, 'Manufacturin...","(2, 'Adm support including clerical') (19.7%),..."


### 1B. Cluster Persona Naming
Assign descriptive, business‑friendly names based on labor participation and income potential.


In [64]:
# Persona naming based on income_prob and weeks worked
cluster_stats = summary[["income_prob_mean", "weeks_worked_mean"]].copy()
order = cluster_stats.sort_values(["weeks_worked_mean", "income_prob_mean"]).index.tolist()

if len(order) == 3:
    name_map = {
        order[0]: "Non-labor-force / low activity",
        order[1]: "Partial-work / early career",
        order[2]: "Full-time workers"
    }
else:
    name_map = {cid: f"Cluster {cid}" for cid in summary.index}

summary["cluster_name"] = summary.index.map(name_map)
df["cluster_name"] = df["cluster"].map(name_map)

display(summary[["cluster_name", "weight_pct", "income_prob_mean", "weeks_worked_mean"]])


,cluster_name,weight_pct,income_prob_mean,weeks_worked_mean
cluster,,,,
1,Non-labor-force / low activity,0.779696,0.066863,21.661617
0,Partial-work / early career,0.166649,0.050512,25.478961
2,Full-time workers,0.053655,0.054678,45.605485


In [65]:
# Step 1: Build segmentation base features
X_full = df.drop(columns=["label", "weight"])

# ensure categorical numeric are strings
for c in categorical_numeric_cols:
    X_full[c] = X_full[c].astype("string")

# add income potential score from final HGB model
df["income_prob"] = final_model.predict_proba(X_full)[:, 1]

# segmentation feature set (exclude label/weight, include income_prob)
X_seg = df.drop(columns=["label", "weight"])

### 2. Preprocessing and Dimensionality Reduction

To prepare for clustering, I applied preprocessing that is appropriate for mixed data types and high‑dimensional categorical features.

- **Numeric features**: imputed with the median and standardized.
- **Categorical features**: imputed with the most frequent value and one‑hot encoded.
- The resulting sparse matrix is high‑dimensional, so I used **TruncatedSVD** to reduce it to a compact embedding (e.g., 30 dimensions), then scaled it again for clustering.

This produces a dense, lower‑dimensional representation that preserves structure while making clustering efficient and stable.


In [66]:
# Step 2: define columns for segmentation
seg_numeric_cols = continuous_cols + ["income_prob"]
seg_categorical_cols = categorical_str_cols + categorical_numeric_cols

# preprocess
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocess_seg = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, seg_numeric_cols),
        ("cat", categorical_transformer, seg_categorical_cols)
    ],
    remainder="drop"
)

# transform -> SVD embedding
X_sparse = preprocess_seg.fit_transform(X_seg)
svd = TruncatedSVD(n_components=30, random_state=42)
X_emb = svd.fit_transform(X_sparse)

# optional scaling after SVD
X_emb = StandardScaler().fit_transform(X_emb)

### 3. Selecting the Number of Clusters

I tested several values of `k` and evaluated them using:

- **Inertia (Elbow method)** to identify diminishing returns in cluster compactness.
- **Silhouette score** (computed on a sample for speed) to measure separation quality.

Based on these diagnostics, I selected the `k` that best balances interpretability and cluster quality.

In [67]:
w_full = df["weight"].values

k_values = [3, 4, 5, 6, 7, 8]
results = []

# sample for silhouette (faster)
rng = np.random.RandomState(42)
sample_size = 20000
sample_idx = rng.choice(len(X_emb), size=sample_size, replace=False)
X_sample = X_emb[sample_idx]

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    km.fit(X_emb, sample_weight=w_full)

    inertia = km.inertia_
    sil = silhouette_score(X_sample, km.predict(X_sample))

    results.append((k, inertia, sil))

print("k, inertia, silhouette")
for r in results:
    print(r)

# choose best k by silhouette (or decide manually from results)
best_k = max(results, key=lambda x: x[2])[0]
print("Best k (silhouette):", best_k)


k, inertia, silhouette
(3, 9801357150.715, 0.12810044640528254)
(4, 9484759600.998955, 0.04417231826989529)
(5, 9291033540.277561, 0.04249299412688804)
(6, 9023688260.445286, 0.053000311671682995)
(7, 8757541480.624773, 0.05857083595134759)
(8, 8479076777.9417, 0.06678496556864312)
Best k (silhouette): 3


### 4. Final KMeans Clustering

Using the chosen `k`, I trained a final **KMeans** model on the full embedded dataset (with sample weights).

Each record was assigned a cluster label and stored in the dataset for downstream profiling and analysis.


In [68]:
final_k = best_k  # or set manually if you prefer
kmeans = KMeans(n_clusters=final_k, random_state=42, n_init="auto")

clusters = kmeans.fit_predict(X_emb, sample_weight=w_full)
df["cluster"] = clusters

print(df["cluster"].value_counts())

cluster
1    154030
0     35144
2     10349
Name: count, dtype: int64


### 5. Cluster Profiling and Interpretation

I summarized each cluster to make the segments interpretable and actionable:

- **Weighted size** and population share
- **Means** of key numeric indicators (age, weeks worked, wages, capital gains, income probability)
- **Top categorical values** (e.g., education, occupation, industry, marital status)

This profiling translates clusters into marketing‑relevant personas and highlights how segments differ in socioeconomic characteristics.

In [ ]:
if "cluster" not in df.columns:
    print("cluster not found — creating it now.")

    # 1) Ensure income_prob exists (needed for segmentation features)
    if "income_prob" not in df.columns:
        if "final_model" not in globals():
            raise ValueError("final_model not found. Run the modeling section before segmentation.")
        X_full = df.drop(columns=["label", "weight"])
        for c in categorical_numeric_cols:
            X_full[c] = X_full[c].astype("string")
        df["income_prob"] = final_model.predict_proba(X_full)[:, 1]

    # 2) Ensure embedding exists
    if "X_emb" not in globals():
        seg_numeric_cols = continuous_cols + ["income_prob"]
        seg_categorical_cols = categorical_str_cols + categorical_numeric_cols

        numeric_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])

        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
        ])

        preprocess_seg = ColumnTransformer(
            transformers=[
                ("num", numeric_transformer, seg_numeric_cols),
                ("cat", categorical_transformer, seg_categorical_cols)
            ],
            remainder="drop"
        )

        X_seg = df.drop(columns=["label", "weight"])
        X_sparse = preprocess_seg.fit_transform(X_seg)
        svd = TruncatedSVD(n_components=30, random_state=42)
        X_emb = svd.fit_transform(X_sparse)
        X_emb = StandardScaler().fit_transform(X_emb)

    # 3) Weights for KMeans
    if "w_full" not in globals():
        w_full = df["weight"].values

    if "best_k" in globals():
        final_k = best_k
    elif "final_k" in globals():
        final_k = final_k
    else:
        final_k = 3
        print("best_k not found — defaulting final_k=3")

    kmeans = KMeans(n_clusters=final_k, random_state=42, n_init="auto")
    df["cluster"] = kmeans.fit_predict(X_emb, sample_weight=w_full)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

def wmean(x, w):
    return np.average(x, weights=w)

# ====== 1) Summary table (weighted means + share) ======
g = df.groupby("cluster")
w = df["weight"]

summary = pd.DataFrame({
    "n": g.size(),
    "weight_sum": g["weight"].sum(),
    "weight_pct": g["weight"].sum() / g["weight"].sum().sum(),
    "income_prob_mean": g.apply(lambda x: wmean(x["income_prob"], x["weight"])),
    "age_mean": g.apply(lambda x: wmean(x["age"], x["weight"])),
    "weeks_worked_mean": g.apply(lambda x: wmean(x["weeks worked in year"], x["weight"])),
    "wage_per_hour_mean": g.apply(lambda x: wmean(x["wage per hour"], x["weight"])),
    "capital_gains_mean": g.apply(lambda x: wmean(x["capital gains"], x["weight"])),
    "dividends_mean": g.apply(lambda x: wmean(x["dividends from stocks"], x["weight"]))
}).sort_values("weight_sum", ascending=False)

summary_fmt = summary.copy()
summary_fmt["weight_pct"] = (summary_fmt["weight_pct"] * 100).map(lambda x: f"{x:.2f}%")
summary_fmt = summary_fmt.round(2)

display(summary_fmt)


,n,weight_sum,weight_pct,income_prob_mean,age_mean,weeks_worked_mean,wage_per_hour_mean,capital_gains_mean,dividends_mean
cluster,,,,,,,,,
1,154030,2.707463e+08,77.97%,0.07,34.50,21.66,0.37,468.81,222.83
0,35144,5.786829e+07,16.66%,0.05,33.60,25.48,13.05,355.63,105.74
2,10349,1.863130e+07,5.37%,0.05,36.77,45.61,1023.26,312.40,114.77


In [ ]:
if "cluster" not in df.columns:
    print("cluster not found — creating it now.")
    if "best_k" in globals():
        final_k = best_k
    elif "final_k" in globals():
        final_k = final_k
    else:
        final_k = 3
        print("best_k not found — defaulting final_k=3")

    kmeans = KMeans(n_clusters=final_k, random_state=42, n_init="auto")
    df["cluster"] = kmeans.fit_predict(X_emb, sample_weight=w_full)

# ====== 2) Top categories per cluster (weighted, readable) ======
def top_cats(df, col, top_n=3):
    tmp = (
        df.groupby(["cluster", col])["weight"].sum()
          .groupby(level=0)
          .apply(lambda s: (s / s.sum()).sort_values(ascending=False).head(top_n))
    )
    # convert to "category (xx.x%)" strings
    out = {}
    for cluster in tmp.index.get_level_values(0).unique():
        s = tmp.loc[cluster]
        out[cluster] = ", ".join([f"{idx} ({val*100:.1f}%)" for idx, val in s.items()])
    return out

key_cat_cols = [
    "education", "class of worker", "marital stat",
    "race", "sex", "major industry code", "major occupation code"
]

topcat_table = pd.DataFrame({
    col: pd.Series(top_cats(df, col)) for col in key_cat_cols
})

display(topcat_table)


,education,class of worker,marital stat,race,sex,major industry code,major occupation code
0,"(0, 'High school graduate') (23.0%), (0, 'Chil...","(0, 'Private') (45.2%), (0, 'Not in universe')...","(0, 'Never married') (41.5%), (0, 'Married-civ...","(0, 'White') (74.0%), (0, 'Black') (11.0%), (0...","(0, 'Female') (50.7%), (0, 'Male') (49.3%)","(0, 'Not in universe or children') (43.3%), (0...","(0, 'Not in universe') (43.3%), (0, 'Other ser..."
1,"(1, 'Children') (25.9%), (1, 'High school grad...","(1, 'Not in universe') (53.9%), (1, 'Private')...","(1, 'Never married') (45.1%), (1, 'Married-civ...","(1, 'White') (84.8%), (1, 'Black') (13.1%), (1...","(1, 'Female') (51.4%), (1, 'Male') (48.6%)","(1, 'Not in universe or children') (54.2%), (1...","(1, 'Not in universe') (54.2%), (1, 'Professio..."
2,"(2, 'High school graduate') (39.7%), (2, 'Some...","(2, 'Private') (86.1%), (2, 'Local government'...","(2, 'Married-civilian spouse present') (51.9%)...","(2, 'White') (83.0%), (2, 'Black') (13.8%), (2...","(2, 'Male') (50.1%), (2, 'Female') (49.9%)","(2, 'Retail trade') (20.2%), (2, 'Manufacturin...","(2, 'Adm support including clerical') (19.7%),..."


In [75]:
topcat_table.to_csv("cluster_top_categories.csv")